# Assignment Solution — Do convergent sites cluster in protein domains?

We test whether the convergent sites identified in Notebook 3 are
enriched in specific prestin domains, particularly the TMD.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
np.random.seed(42)

DATA = os.path.join('..', 'data')

---
## 1. Load convergence results

In [ ]:
results_df = pd.read_csv(os.path.join(DATA, 'prestin_convergence_results.csv'))
print(f"Total alignment positions: {len(results_df)}")
results_df.head()

---
## 2. Identify convergent sites

In [ ]:
threshold = 0.05
convergent = results_df[results_df['pvalue'] < threshold]
convergent_positions = convergent['position'].values
n_convergent = len(convergent_positions)
n_total = len(results_df)

print(f"Convergent sites (p < {threshold}): {n_convergent} / {n_total}")

---
## 3. Define domains and count

In [ ]:
DOMAINS = {
    'N-terminal': (0, 79),
    'TMD':        (80, 504),
    'Linker':     (505, 529),
    'STAS':       (530, n_total - 1),
}

print(f"{'Domain':12s}  {'Positions':>10s}  {'% of aln':>8s}  "
      f"{'Conv sites':>10s}  {'% of conv':>9s}  {'Fold':>5s}")
print("-" * 65)

for name, (start, end) in DOMAINS.items():
    n_domain = end - start + 1
    n_in = sum(1 for p in convergent_positions if start <= p <= end)
    frac_pos = n_domain / n_total
    frac_conv = n_in / n_convergent if n_convergent > 0 else 0
    fold = frac_conv / frac_pos if frac_pos > 0 else 0
    print(f"{name:12s}  {n_domain:10d}  {frac_pos:8.1%}  "
          f"{n_in:10d}  {frac_conv:9.1%}  {fold:5.2f}")

---
## 4. Permutation test — TMD enrichment

**Test statistic:** number of convergent sites falling in the TMD.

**Null hypothesis:** convergent sites are randomly distributed
across the alignment. Under the null, we randomly sample
`n_convergent` positions from all `n_total` positions and count
how many fall in the TMD.

**What we shuffle:** positions (not species labels). We ask: if
we picked this many random positions from the alignment, how often
would at least this many land in the TMD?

In [ ]:
tmd_start, tmd_end = DOMAINS['TMD']

# Observed statistic
obs_in_tmd = sum(1 for p in convergent_positions
                 if tmd_start <= p <= tmd_end)
print(f"Observed convergent sites in TMD: {obs_in_tmd}")

# Permutation test
n_perm = 10000
null_counts = np.zeros(n_perm)

for i in range(n_perm):
    random_pos = np.random.choice(n_total, size=n_convergent, replace=False)
    null_counts[i] = np.sum((random_pos >= tmd_start) & (random_pos <= tmd_end))

# P-value for enrichment (one-sided: observed >= null)
p_enrichment = np.mean(null_counts >= obs_in_tmd)
print(f"Expected by chance: {np.mean(null_counts):.1f}")
print(f"P-value (TMD enrichment): {p_enrichment:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null_counts, bins=range(int(null_counts.min()),
        int(null_counts.max()) + 2), color='steelblue',
        alpha=0.7, edgecolor='white')
ax.axvline(obs_in_tmd, color='red', linewidth=2, linestyle='--',
           label=f'Observed = {obs_in_tmd}')
ax.axvline(np.mean(null_counts), color='gray', linewidth=1,
           linestyle=':', label=f'Expected = {np.mean(null_counts):.1f}')
ax.set_xlabel('Convergent sites in TMD')
ax.set_ylabel('Count')
ax.set_title(f'TMD enrichment test — p = {p_enrichment:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

The observed number of convergent sites in the TMD is **below** the
expected value — the TMD is not enriched, it may even be slightly
depleted.

---
## 5. Check other domains

Since the TMD is not enriched, let's test all domains.

In [ ]:
print(f"{'Domain':12s}  {'Obs':>4s}  {'Exp':>5s}  {'Fold':>5s}  {'p (enrich)':>10s}")
print("-" * 45)

for name, (start, end) in DOMAINS.items():
    obs = sum(1 for p in convergent_positions if start <= p <= end)

    null = np.zeros(n_perm)
    for i in range(n_perm):
        rp = np.random.choice(n_total, size=n_convergent, replace=False)
        null[i] = np.sum((rp >= start) & (rp <= end))

    p_val = np.mean(null >= obs)
    expected = np.mean(null)
    fold = obs / expected if expected > 0 else 0
    print(f"{name:12s}  {obs:4d}  {expected:5.1f}  {fold:5.2f}  {p_val:10.4f}")

---
## 6. Visualize convergent site distribution

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

# Color background by domain
colors = {'N-terminal': '#2196F3', 'TMD': '#F44336',
          'Linker': '#FF9800', 'STAS': '#4CAF50'}

for name, (start, end) in DOMAINS.items():
    ax.axvspan(start, end, alpha=0.15, color=colors[name], label=name)

# Mark convergent sites
for p in convergent_positions:
    ax.axvline(p, color='black', alpha=0.5, linewidth=0.8)

ax.set_xlim(0, n_total)
ax.set_xlabel('Alignment position')
ax.set_title('Convergent sites across prestin domains')
ax.legend(loc='upper right')
ax.set_yticks([])
plt.tight_layout()
plt.show()

---
## 7. Interpretation

Convergent sites are **not** significantly enriched in the TMD
(motor domain). If anything, the TMD shows a slight depletion
(fold ≈ 0.7), while the N-terminal and STAS domains show
non-significant trends toward enrichment.

This suggests that the molecular convergence underlying echolocation
in prestin is not concentrated in the transmembrane motor domain.
Instead, convergent substitutions appear distributed across the
protein, possibly including regulatory regions (STAS, N-terminal).
This is consistent with the view that functional convergence can
arise from changes in regulation and protein-protein interactions,
not just the core catalytic or mechanical domain.